# Semantic Correspondence — Complete Analysis Report

This notebook analyzes all results produced by the Google Colab pipeline.  
It runs **fully local on Mac**, without GPU.

## Setup
1. Download this folder from Google Drive:  
   `MyDrive/Colab Notebooks/AML_results/runs/pipeline_exports/`
2. Place it next to this notebook (or update `EXPORTS_DIR` below).
3. Install dependencies once:
   ```bash
   pip install pandas matplotlib seaborn numpy
   ```
4. Run all cells: **Cell → Run All**.

Figures are automatically saved into `./figures/`.

---
**Report sections**
1. Data loading · 2. Aggregate table · 3. Stage progression  
4. Backbone comparison · 5. Fine-tuning depth sweep · 6. WSA impact  
7. LoRA vs Fine-tuning · 8. Per-category PCK · 9. Difficulty analysis · 10. Summary

In [ ]:
from pathlib import Path

# ── Update this path to the folder downloaded from Drive ─────────────────────
EXPORTS_DIR = Path("./pipeline_exports")
# ──────────────────────────────────────────────────────────────────────────────

FIGURES_DIR = Path("./figures")
FIGURES_DIR.mkdir(exist_ok=True)

print(f"EXPORTS_DIR : {EXPORTS_DIR.resolve()}")
print(f"Exists      : {EXPORTS_DIR.exists()}")
if EXPORTS_DIR.exists():
    print(f"Files found : {[f.name for f in sorted(EXPORTS_DIR.iterdir())]}")

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded.")

## 1 · Data Loading

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print(f"[SKIP] File not found: {path.name}")
        return None
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"[OK]   {path.name}  ({len(data)} entries)")
    return data

results          = load_json(EXPORTS_DIR / "pck_results.json") or []
per_category_raw = load_json(EXPORTS_DIR / "pck_results_per_category.json") or []
difficulty_raw   = load_json(EXPORTS_DIR / "pck_results_by_difficulty_flag.json") or []
per_image_raw    = load_json(EXPORTS_DIR / "pck_results_per_image.json") or []

print(f"\nTotal evaluation runs: {len(results)}")

In [ ]:
# ── Global constants and run-name parsing ───────────────────────────────────
BACKBONES = ["dinov2_vitb14", "dinov3_vitb16", "sam_vit_b"]
BB_LABEL  = {
    "dinov2_vitb14": "DINOv2 ViT-B/14",
    "dinov3_vitb16": "DINOv3 ViT-B/16",
    "sam_vit_b":     "SAM ViT-B",
}
BB_COLOR = {
    "dinov2_vitb14": "#2196F3",
    "dinov3_vitb16": "#FF9800",
    "sam_vit_b":     "#4CAF50",
}
SPAIR_CATS = [
    "aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car",
    "cat", "chair", "cow", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "train", "tvmonitor",
]
FLAGS = ["viewpoint_variation", "scale_variation", "truncation", "occlusion"]
FLAG_LABELS = {
    "viewpoint_variation": "Viewpoint\nvariation",
    "scale_variation":     "Scale\nvariation",
    "truncation":          "Truncation",
    "occlusion":           "Occlusion",
}

def parse_run(name: str) -> dict:
    """Parse run name into backbone, stage, blocks, and WSA flag."""
    for bb in BACKBONES:
        if name == bb or name.startswith(bb + "_"):
            rest = name[len(bb):].lstrip("_")
            wsa  = rest.endswith("_wsa") or rest == "wsa"
            if wsa:
                rest = rest[:-4].rstrip("_") if rest != "wsa" else ""
            if rest in ("", "baseline"):
                return dict(backbone=bb, stage="baseline", blocks=None, wsa=wsa)
            if rest.startswith("ft_lb"):
                try:
                    return dict(backbone=bb, stage="finetune", blocks=int(rest[5:]), wsa=wsa)
                except ValueError:
                    pass
            if rest == "lora":
                return dict(backbone=bb, stage="lora", blocks=None, wsa=wsa)
    return dict(backbone=name, stage="unknown", blocks=None, wsa=False)

def stage_label(stage, blocks=None):
    return {"baseline": "Baseline",
            "finetune": f"Fine-tuning (lb={int(blocks)})" if blocks else "Fine-tuning",
            "lora":     "LoRA"}.get(stage, stage)

def run_label(name):
    p = parse_run(name)
    bb  = BB_LABEL.get(p["backbone"], p["backbone"])
    s   = stage_label(p["stage"], p["blocks"])
    wsa = " + WSA" if p["wsa"] else ""
    return f"{bb} | {s}{wsa}"

# ── Main DataFrame ───────────────────────────────────────────────────────────
rows = []
for r in results:
    p = parse_run(r["name"])
    m = r.get("metrics", {})
    rows.append({
        "name":     r["name"],
        "backbone": p["backbone"],
        "stage":    p["stage"],
        "blocks":   p["blocks"],
        "wsa":      p["wsa"],
        "pck@0.05": m.get("pck@0.05", float("nan")),
        "pck@0.1":  m.get("pck@0.1",  float("nan")),
        "pck@0.2":  m.get("pck@0.2",  float("nan")),
    })

df = pd.DataFrame(rows)
backbones_present = [b for b in BACKBONES if b in df["backbone"].values]
STAGE_SORT = {"baseline": 0, "finetune": 1, "lora": 2, "unknown": 9}

print(f"DataFrame: {df.shape[0]} runs x {df.shape[1]} columns")
print(f"Backbones present: {backbones_present}")
print("\nRuns found:")
for n in df["name"].tolist():
    print(f"  {n}")

## 2 · Full Aggregate Table
PCK@{0.05, 0.1, 0.2} for all runs. Colors scale from white (lower) to green (higher) per column.

In [ ]:
df_sorted = df.copy()
df_sorted["_sbb"]  = df_sorted["backbone"].map(lambda b: BACKBONES.index(b) if b in BACKBONES else 9)
df_sorted["_ss"]   = df_sorted["stage"].map(STAGE_SORT)
df_sorted["_sblk"] = df_sorted["blocks"].fillna(0)
df_sorted["_swsa"] = df_sorted["wsa"].astype(int)
df_sorted = df_sorted.sort_values(["_sbb", "_ss", "_sblk", "_swsa"])
df_sorted["Method"] = df_sorted["name"].map(run_label)

table = df_sorted[["Method", "pck@0.05", "pck@0.1", "pck@0.2"]].set_index("Method")
table.columns = ["PCK@0.05", "PCK@0.10", "PCK@0.20"]

styled = (
    table.style
    .format("{:.4f}")
    .background_gradient(cmap="YlGn", axis=0)
    .set_caption("PCK@alpha — all methods (higher is better)")
    .set_table_styles([{"selector": "caption",
                        "props": [("font-size", "14px"), ("font-weight", "bold")]}])
)
display(styled)

table.to_csv(FIGURES_DIR / "pck_table_full.csv")
print("Saved: figures/pck_table_full.csv")

## 3 · Stage Progression
For each backbone: how PCK@0.1 changes from Baseline -> Best Fine-tuning -> LoRA (with and without WSA).

In [ ]:
METRIC = "pck@0.1"

stage_steps = [
    ("baseline", False, "Baseline"),
    ("baseline", True,  "Baseline\n+WSA"),
    ("finetune", False, "Best FT"),
    ("finetune", True,  "Best FT\n+WSA"),
    ("lora",     False, "LoRA"),
    ("lora",     True,  "LoRA\n+WSA"),
]

fig, axes = plt.subplots(1, len(backbones_present),
                         figsize=(5 * len(backbones_present), 5), sharey=True)
if len(backbones_present) == 1:
    axes = [axes]

for ax, bb in zip(axes, backbones_present):
    sub = df[df["backbone"] == bb]
    vals, xlbls = [], []
    for stage, wsa, lbl in stage_steps:
        s2 = sub[(sub["stage"] == stage) & (sub["wsa"] == wsa)]
        if s2.empty or s2[METRIC].isna().all():
            continue
        vals.append(s2[METRIC].max())  # Best variant (for example, best block count).
        xlbls.append(lbl)

    x = np.arange(len(vals))
    alphas_bar = [0.95 if i % 2 == 0 else 0.5 for i in range(len(vals))]
    bars = ax.bar(x, vals, color=BB_COLOR[bb], edgecolor="white", linewidth=1.2,
                  alpha=1.0)  # Use face alpha to distinguish +WSA variants.
    for i, (bar, v, ab) in enumerate(zip(bars, vals, alphas_bar)):
        bar.set_alpha(ab)
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xticks(x)
    ax.set_xticklabels(xlbls, fontsize=9)
    ax.set_title(BB_LABEL[bb], fontweight="bold")
    ax.set_xlabel("Stage")
    ax.set_ylabel(METRIC.upper())
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax.text(0.97, 0.03, "lighter = +WSA", transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, color="gray")

fig.suptitle("PCK@0.1 progression by stage — all backbones",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stage_progression.png", bbox_inches="tight")
plt.show()
print("Saved: figures/stage_progression.png")

## 4 · Backbone Comparison — Best Method per Backbone
For each backbone, show the run with the highest PCK@0.1 across all three alpha thresholds.

In [ ]:
alphas = ["pck@0.05", "pck@0.1", "pck@0.2"]
alpha_labels = ["PCK@0.05", "PCK@0.10", "PCK@0.20"]

best_per_bb = {}
for bb in backbones_present:
    sub = df[df["backbone"] == bb]
    if sub.empty:
        continue
    best_per_bb[bb] = sub.loc[sub["pck@0.1"].idxmax()]

x = np.arange(len(alphas))
width = 0.7 / max(len(best_per_bb), 1)
fig, ax = plt.subplots(figsize=(9, 5))

for i, (bb, row) in enumerate(best_per_bb.items()):
    vals   = [row[a] for a in alphas]
    offset = (i - len(best_per_bb) / 2 + 0.5) * width
    bars   = ax.bar(x + offset, vals, width * 0.9,
                    label=f"{BB_LABEL[bb]}\n({row['name']})",
                    color=BB_COLOR[bb], edgecolor="white", linewidth=1)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(alpha_labels, fontsize=11)
ax.set_ylabel("PCK@alpha")
ax.set_title("Best configuration per backbone (all alpha thresholds)", fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "backbone_comparison.png", bbox_inches="tight")
plt.show()
print("Saved: figures/backbone_comparison.png")

## 5 · Fine-Tuning Depth Sweep
`last_blocks in {1, 2, 4}` — number of transformer blocks unfrozen during fine-tuning.

In [ ]:
ft_df = df[df["stage"] == "finetune"].copy()

if ft_df.empty:
    print("No fine-tuning runs found.")
else:
    fig, axes = plt.subplots(1, len(backbones_present),
                             figsize=(5 * len(backbones_present), 5), sharey=True)
    if len(backbones_present) == 1:
        axes = [axes]

    for ax, bb in zip(axes, backbones_present):
        sub = ft_df[ft_df["backbone"] == bb].sort_values("blocks")
        if sub.empty:
            ax.set_title(BB_LABEL[bb] + "\n(no data)")
            continue

        blocks_vals = sorted(sub["blocks"].dropna().unique())
        x = np.arange(len(blocks_vals))
        w = 0.35

        for offset, wsa, alpha_bar, lbl in [
            (-w / 2, False, 0.95, "without WSA"),
            ( w / 2, True,  0.50, "with WSA"),
        ]:
            s2 = sub[sub["wsa"] == wsa]
            vals_per_blk = [
                s2[s2["blocks"] == b]["pck@0.1"].values[0]
                if not s2[s2["blocks"] == b].empty else float("nan")
                for b in blocks_vals
            ]
            bars = ax.bar(x + offset, vals_per_blk, w,
                          label=lbl, color=BB_COLOR[bb],
                          alpha=alpha_bar, edgecolor="white")
            for bar, v in zip(bars, vals_per_blk):
                if not np.isnan(v):
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.002,
                            f"{v:.3f}", ha="center", va="bottom", fontsize=8.5)

        ax.set_xticks(x)
        ax.set_xticklabels([f"lb={int(b)}" for b in blocks_vals])
        ax.set_title(BB_LABEL[bb], fontweight="bold")
        ax.set_ylabel("PCK@0.1")
        ax.legend(fontsize=9)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

    fig.suptitle("Impact of unfrozen block count (fine-tuning depth sweep)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "finetune_depth_sweep.png", bbox_inches="tight")
    plt.show()
    print("Saved: figures/finetune_depth_sweep.png")

## 6 · Window Soft-Argmax (WSA) Impact
Delta PCK = (with WSA) - (without WSA) for each run. Positive values mean WSA helps.

In [ ]:
no_wsa  = df[df["wsa"] == False].copy()
yes_wsa = df[df["wsa"] == True].copy()

merged = no_wsa.merge(
    yes_wsa[["backbone", "stage", "blocks", "pck@0.05", "pck@0.1", "pck@0.2"]],
    on=["backbone", "stage", "blocks"],
    suffixes=("_base", "_wsa"),
)

if merged.empty:
    print("No (base, +WSA) pairs found.")
else:
    for a in ["pck@0.05", "pck@0.1", "pck@0.2"]:
        merged[f"delta_{a}"] = merged[f"{a}_wsa"] - merged[f"{a}_base"]

    merged_s = merged.sort_values(["backbone", "stage", "blocks"])

    def short_label(row):
        s = {"baseline": "Baseline",
             "finetune": f"FT lb={int(row['blocks'])}" if row["blocks"] else "FT",
             "lora": "LoRA"}.get(row["stage"], row["stage"])
        bb = BB_LABEL.get(row["backbone"], row["backbone"]).replace(" ", "\n")
        return f"{bb}\n{s}"

    xlabels = [short_label(r) for _, r in merged_s.iterrows()]
    colors_b = [BB_COLOR.get(r["backbone"], "gray") for _, r in merged_s.iterrows()]
    x = np.arange(len(merged_s))
    w = 0.22

    fig, ax = plt.subplots(figsize=(max(10, len(merged_s) * 1.1), 5))
    for offset, a, alpha_v, lbl in [
        (-w, "pck@0.05", 0.5, "Delta PCK@0.05"),
        ( 0, "pck@0.1",  0.9, "Delta PCK@0.10"),
        ( w, "pck@0.2",  0.6, "Delta PCK@0.20"),
    ]:
        deltas = merged_s[f"delta_{a}"].values
        ax.bar(x + offset, deltas, w, label=lbl,
               color=colors_b, alpha=alpha_v, edgecolor="white")

    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=8.5)
    ax.set_ylabel("Delta PCK  (WSA - base)")
    ax.set_title("WSA impact on each method", fontweight="bold")

    bb_patches = [mpatches.Patch(color=BB_COLOR[b], label=BB_LABEL[b])
                  for b in backbones_present]
    alpha_handles, alpha_lbls = ax.get_legend_handles_labels()
    ax.legend(handles=bb_patches + alpha_handles[:3],
              labels=[BB_LABEL[b] for b in backbones_present] + alpha_lbls[:3],
              fontsize=9, ncol=2, loc="upper left")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "wsa_impact.png", bbox_inches="tight")
    plt.show()

    print("\nMean WSA delta by stage (PCK@0.1):")
    print(merged_s.groupby(["backbone", "stage"])["delta_pck@0.1"].mean().round(4))
    print("Saved: figures/wsa_impact.png")

## 7 · LoRA vs Fine-Tuning
Direct comparison among Baseline, best Fine-tuning, and LoRA for each backbone, with and without WSA.

In [ ]:
groups = [
    ("baseline", False, "Baseline"),
    ("baseline", True,  "Baseline\n+WSA"),
    ("finetune", False, "Best FT"),
    ("finetune", True,  "Best FT\n+WSA"),
    ("lora",     False, "LoRA"),
    ("lora",     True,  "LoRA+WSA"),
]
METRIC_PLOT = "pck@0.1"

x     = np.arange(len(groups))
n_bb  = len(backbones_present)
width = 0.75 / max(n_bb, 1)

fig, ax = plt.subplots(figsize=(12, 5))
for i, bb in enumerate(backbones_present):
    sub = df[df["backbone"] == bb]
    vals = []
    for stage, wsa, _ in groups:
        s2 = sub[(sub["stage"] == stage) & (sub["wsa"] == wsa)]
        vals.append(s2[METRIC_PLOT].max() if not s2.empty else float("nan"))

    offset = (i - n_bb / 2 + 0.5) * width
    bar_alphas = [1.0 if j % 2 == 0 else 0.55 for j in range(len(vals))]
    bars = ax.bar(x + offset, vals, width * 0.92,
                  color=BB_COLOR[bb], label=BB_LABEL[bb], edgecolor="white")
    for bar, v, ab in zip(bars, vals, bar_alphas):
        bar.set_alpha(ab)
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.001,
                    f"{v:.3f}", ha="center", va="bottom",
                    fontsize=7.5, rotation=40)

ax.set_xticks(x)
ax.set_xticklabels([lbl for _, _, lbl in groups], fontsize=10)
ax.set_ylabel(METRIC_PLOT.upper())
ax.set_title("Method comparison by backbone — PCK@0.1  (lighter = +WSA)",
             fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lora_vs_finetune.png", bbox_inches="tight")
plt.show()
print("Saved: figures/lora_vs_finetune.png")

## 8 · Per-Category PCK (Heatmap)
Each row is one method, each column is one of the 18 SPair-71k categories, and values are PCK@0.1.  
Only runs without WSA are shown to reduce redundancy.

In [ ]:
if not per_category_raw:
    print("File pck_results_per_category.json not found — skipping section.")
else:
    ALPHA_CAT = "pck@0.1"

    # Build matrix (run x category)
    cat_records = {}
    for entry in per_category_raw:
        row_d = {}
        for cat in SPAIR_CATS:
            row_d[cat] = entry["categories"].get(cat, {}).get(ALPHA_CAT, float("nan"))
        cat_records[entry["name"]] = row_d

    cat_df = pd.DataFrame(cat_records).T   # run x category

    # Sort by backbone/stage and keep only runs without WSA
    no_wsa_names = df_sorted[~df_sorted["wsa"]]["name"].tolist()
    cat_df = cat_df.loc[[n for n in no_wsa_names if n in cat_df.index]]

    if cat_df.empty:
        # Fallback: show all runs
        cat_df = pd.DataFrame(cat_records).T

    cat_df.index = [run_label(n) for n in cat_df.index]

    fig, ax = plt.subplots(figsize=(max(14, len(SPAIR_CATS) * 0.82),
                                    max(6, len(cat_df) * 0.5)))
    sns.heatmap(
        cat_df.astype(float),
        ax=ax,
        cmap="YlOrRd",
        annot=True,
        fmt=".2f",
        annot_kws={"size": 7.5},
        linewidths=0.4,
        cbar_kws={"label": "PCK@0.1"},
        vmin=0.0,
    )
    ax.set_title("PCK@0.1 per SPair-71k category",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Category")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=40, labelsize=9)
    ax.tick_params(axis="y", labelsize=8.5)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "pck_per_category_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Saved: figures/pck_per_category_heatmap.png")

    print("\nCategory ranking by mean PCK@0.1 (across all methods):")
    print(cat_df.mean().sort_values(ascending=False).round(4).to_string())

## 9 · Difficulty Flag Analysis
SPair-71k labels each pair with 4 binary flags.  
Compare PCK@0.1 when a flag is **absent (0)** vs **present (1)** for the best run of each backbone.

In [ ]:
if not difficulty_raw:
    print("File pck_results_by_difficulty_flag.json not found — skipping section.")
else:
    # Best run per backbone (based on PCK@0.1)
    best_names = {}
    for bb in backbones_present:
        sub = df[df["backbone"] == bb]
        if not sub.empty:
            best_names[bb] = sub.loc[sub["pck@0.1"].idxmax(), "name"]

    diff_by_name = {r["name"]: r for r in difficulty_raw}
    runs_diff = {bb: diff_by_name[n]
                 for bb, n in best_names.items() if n in diff_by_name}

    if not runs_diff:
        print("No runs compatible with difficulty data.")
    else:
        fig, axes = plt.subplots(1, len(FLAGS),
                                 figsize=(4.5 * len(FLAGS), 5), sharey=False)

        for ax, flag in zip(axes, FLAGS):
            data_rows = []
            for bb, entry in runs_diff.items():
                flag_data = entry.get("data", {}).get(flag, {})
                for bucket in ["0", "1"]:
                    summary = flag_data.get(bucket, {}).get("summary", {})
                    v = summary.get("image", {}).get("custom_pck0.1", {}).get("all",
                        float("nan"))
                    data_rows.append({"backbone": bb, "bucket": int(bucket), "pck@0.1": v})

            flag_df = pd.DataFrame(data_rows)
            bbs     = [b for b in backbones_present if b in flag_df["backbone"].values]
            x       = np.arange(len(bbs))
            w       = 0.35

            for offset, bucket, alpha_v, lbl in [
                (-w / 2, 0, 0.95, "Absent (0)"),
                ( w / 2, 1, 0.45, "Present (1)"),
            ]:
                sub = flag_df[flag_df["bucket"] == bucket]
                vals = [sub[sub["backbone"] == b]["pck@0.1"].values[0]
                        if not sub[sub["backbone"] == b].empty else float("nan")
                        for b in bbs]
                bars = ax.bar(x + offset, vals, w,
                              color=[BB_COLOR.get(b, "gray") for b in bbs],
                              alpha=alpha_v, label=lbl, edgecolor="white")
                for bar, v in zip(bars, vals):
                    if not np.isnan(v):
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.002,
                                f"{v:.3f}", ha="center", va="bottom", fontsize=8)

            ax.set_xticks(x)
            ax.set_xticklabels([BB_LABEL.get(b, b) for b in bbs],
                               fontsize=8.5, rotation=15)
            ax.set_title(FLAG_LABELS[flag], fontweight="bold")
            ax.set_ylabel("PCK@0.1")
            ax.legend(fontsize=8)

        fig.suptitle(
            "PCK@0.1 by difficulty flag — best model per backbone",
            fontsize=13, fontweight="bold", y=1.02
        )
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "difficulty_analysis.png", bbox_inches="tight")
        plt.show()
        print("Saved: figures/difficulty_analysis.png")

## 10 · Summary — Best Models

In [ ]:
print("=" * 72)
print("  SUMMARY — BEST CONFIGURATION BY BACKBONE x METRIC")
print("=" * 72)

summary_rows = []
for bb in backbones_present:
    sub = df[df["backbone"] == bb]
    if sub.empty:
        continue
    for metric in ["pck@0.05", "pck@0.1", "pck@0.2"]:
        best = sub.loc[sub[metric].idxmax()]
        summary_rows.append({
            "Backbone": BB_LABEL[bb],
            "Metric":   metric.upper(),
            "Value":    round(float(best[metric]), 5),
            "Run":      best["name"],
        })

summary_df = pd.DataFrame(summary_rows)
display(
    summary_df.style
    .format({"Value": "{:.5f}"})
    .background_gradient(subset=["Value"], cmap="Greens")
    .set_caption("Best configuration by backbone x metric")
)

best_overall = df.loc[df["pck@0.1"].idxmax()]
print(f"\nBest overall result (PCK@0.1):")
print(f"  Run     : {best_overall['name']}")
print(f"  PCK@0.05: {best_overall['pck@0.05']:.4f}")
print(f"  PCK@0.10: {best_overall['pck@0.1']:.4f}")
print(f"  PCK@0.20: {best_overall['pck@0.2']:.4f}")

summary_df.to_csv(FIGURES_DIR / "summary_best_models.csv", index=False)
print("\nSaved: figures/summary_best_models.csv")

## 11 · Export Output Files

In [ ]:
# Full sorted table as report-ready CSV
df_export = df_sorted[["name", "backbone", "stage", "blocks", "wsa",
                        "pck@0.05", "pck@0.1", "pck@0.2"]].copy()
df_export["backbone"] = df_export["backbone"].map(lambda b: BB_LABEL.get(b, b))
df_export["stage"]    = df_export["stage"].map(
    {"baseline": "Baseline", "finetune": "Fine-tuning", "lora": "LoRA"})
df_export.rename(columns={
    "pck@0.05": "PCK@0.05", "pck@0.1": "PCK@0.10", "pck@0.2": "PCK@0.20",
    "blocks": "last_blocks", "wsa": "WSA",
}, inplace=True)
df_export.to_csv(FIGURES_DIR / "all_results_sorted.csv", index=False)

print("\nFiles generated in ./figures/:")
for f in sorted(FIGURES_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:50s}  {size:6.1f} KB")